In [4]:
import pandas as pd

In [5]:
import pandas as pd
print("Pandas working ✔")

Pandas working ✔


In [6]:
import os

DATA_DIR = "FoodData_Central_sr_legacy_food_csv_2018-04"

KEY_NUTRIENTS = {1003: "protein", 1004: "fat", 1005: "carbs", 1008: "calories"}

def build_food_db():
    foods          = pd.read_csv(f"{DATA_DIR}/food.csv")
    food_nutrients = pd.read_csv(f"{DATA_DIR}/food_nutrient.csv")
    food_nutrients["nutrient_id"] = food_nutrients["nutrient_id"].astype(int)
    filtered = food_nutrients[food_nutrients["nutrient_id"].isin(KEY_NUTRIENTS)].copy()
    filtered["nutrient_name"] = filtered["nutrient_id"].map(KEY_NUTRIENTS)
    pivoted = filtered.pivot_table(
        index="fdc_id", columns="nutrient_name", values="amount", aggfunc="first"
    ).reset_index()
    result = pivoted.merge(foods[["fdc_id", "description"]], on="fdc_id")
    result = result.dropna(subset=["calories", "protein", "fat", "carbs"])
    result = result[result["calories"] > 0].reset_index(drop=True)
    result = result.rename(columns={"description": "food_name"})
    result["food_name"] = result["food_name"].str.lower()
    return result[["food_name", "calories", "protein", "carbs", "fat"]]

df = build_food_db()
print(f"Loaded {len(df)} foods")
df.head()

Loaded 7756 foods


,food_name,calories,protein,carbs,fat
0,"pillsbury golden layer buttermilk biscuits, ar...",307.0,5.88,41.18,13.24
1,"pillsbury, cinnamon rolls with icing, refriger...",330.0,4.34,53.42,11.27
2,"kraft foods, shake n bake original recipe, coa...",377.0,6.10,79.80,3.70
3,"george weston bakeries, thomas english muffins",232.0,8.00,46.00,1.80
4,"waffles, buttermilk, frozen, ready-to-heat",273.0,6.58,41.05,9.22


In [7]:
class FoodDatabaseAgent:

    def __init__(self, dataframe):
        self.df = dataframe.copy()
        self.df["food_name"] = self.df["food_name"].str.lower()
        self.df = self.df.dropna(subset=["calories", "protein", "carbs", "fat"])

    def get_food(self, name):
        name = name.lower()
        matches = self.df[self.df["food_name"].str.contains(name, na=False)]

        if matches.empty:
            return None
        
        return matches.iloc[0].to_dict()

    def calculate_macros(self, name, grams):
        food = self.get_food(name)
        if not food:
            return None
        
        factor = grams / 100
        
        return {
            "food_name": food["food_name"],
            "grams": grams,
            "calories": round(food["calories"] * factor, 2),
            "protein": round(food["protein"] * factor, 2),
            "carbs": round(food["carbs"] * factor, 2),
            "fat": round(food["fat"] * factor, 2)
        }

    def calculate_meal(self, meal_items):
        total = {"calories": 0, "protein": 0, "carbs": 0, "fat": 0}
        details = []

        for name, grams in meal_items:
            macros = self.calculate_macros(name, grams)
            if not macros:
                continue

            details.append(macros)

            total["calories"] += macros["calories"]
            total["protein"] += macros["protein"]
            total["carbs"] += macros["carbs"]
            total["fat"] += macros["fat"]

        return {
            "meal_details": details,
            "total_macros": total
        }

In [8]:
agent = FoodDatabaseAgent(df)

In [9]:
agent.calculate_macros("egg", 200)

{'food_name': 'fish, whitefish, eggs (alaska native)',
 'grams': 200,
 'calories': 208.0,
 'protein': 29.32,
 'carbs': 9.78,
 'fat': 5.76}

In [10]:
meal = [
    ("egg", 200),
    ("rice", 150),
    ("chicken", 250)
]

agent.calculate_meal(meal)

{'meal_details': [{'food_name': 'fish, whitefish, eggs (alaska native)',
   'grams': 200,
   'calories': 208.0,
   'protein': 29.32,
   'carbs': 9.78,
   'fat': 5.76},
  {'food_name': 'snacks, rice cakes, brown rice, sesame seed',
   'grams': 150,
   'calories': 588.0,
   'protein': 11.4,
   'carbs': 122.25,
   'fat': 5.7},
  {'food_name': 'restaurant, latino, chicken and rice, entree, prepared',
   'grams': 250,
   'calories': 435.0,
   'protein': 30.05,
   'carbs': 50.08,
   'fat': 12.65}],
 'total_macros': {'calories': 1231.0,
  'protein': 70.77,
  'carbs': 182.11,
  'fat': 24.11}}